# Decoradores de métodos en Python

Ejemplos prácticos de `@classmethod`, `@staticmethod`, `@property`, `functools.cached_property` y cómo aplicar decoradores a métodos.

## 1) `@classmethod`

`@classmethod` convierte un método en uno que recibe la clase (`cls`) como primer argumento en lugar de la instancia. Se usa frecuentemente para **constructores alternativos** o métodos que operan sobre la clase en sí.

In [ ]:
class Persona:
    def __init__(self, nombre, edad):
        self.nombre = nombre
        self.edad = edad

    @classmethod
    def desde_cadena(cls, cadena):
        # formato: "Nombre,edad"
        nombre, edad = cadena.split(",")
        return cls(nombre.strip(), int(edad.strip()))

p = Persona.desde_cadena("Miguel, 30")
print(type(p), p.nombre, p.edad)

## 2) `@staticmethod`

`@staticmethod` define un método que no recibe ni `self` ni `cls`. Es una función agrupada dentro de la clase por conveniencia semántica.

In [ ]:
class Utilidades:
    @staticmethod
    def sumar(a, b):
        return a + b

print(Utilidades.sumar(2, 3))
u = Utilidades()
print(u.sumar(5, 7))  # funciona igual desde la instancia

## 3) `@property` y `@prop.setter`

`@property` permite acceder a un método como si fuera un atributo (getter). Puedes definir setter y deleter con `@<prop>.setter` y `@<prop>.deleter`.

In [ ]:
class Circulo:
    def __init__(self, radio):
        self._radio = radio

    @property
    def radio(self):
        return self._radio

    @radio.setter
    def radio(self, valor):
        if valor <= 0:
            raise ValueError("El radio debe ser mayor que 0")
        self._radio = valor

    @property
    def area(self):
        import math
        return math.pi * self._radio ** 2

c = Circulo(2)
print("radio:", c.radio)
print("area:", c.area)
c.radio = 3
print("nuevo radio:", c.radio, "nueva area:", c.area)

## 4) `functools.cached_property` (Python 3.8+)

`cached_property` calcula el valor la primera vez y lo guarda en la instancia; llamadas posteriores devuelven el valor cacheado sin volver a calcularlo.

In [ ]:
import time
from functools import cached_property

class DatosPesados:
    def __init__(self, x):
        self.x = x

    @cached_property
    def resultado(self):
        time.sleep(1.5)  # simulamos cálculo pesado
        return self.x * 2

d = DatosPesados(10)
import time as _t
_start = _t.time()
print(d.resultado)   # cálculo
print("t1:", _t.time() - _start)
_start = _t.time()
print(d.resultado)   # valor cacheado, rápido
print("t2:", _t.time() - _start)

## 5) Decoradores sobre métodos (preservando `self` y firmas)

Un decorador que envuelve métodos debe aceptar `*args` y `**kwargs`; para preservar metadata conviene usar `functools.wraps`.

In [ ]:
from functools import wraps

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # si es un método, args[0] es self
        owner = args[0].__class__.__name__ if args else None
        print(f"Calling {func.__name__} of {owner} with args={args[1:] if owner else args} kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

class Calculadora:
    @log_calls
    def suma(self, a, b):
        return a + b

calc = Calculadora()
print("resultado:", calc.suma(3, 4))

## 6) Aplicando decoradores junto a `@classmethod` / `@staticmethod`

El orden de los decoradores importa. Aquí un ejemplo que demuestra cómo aplicar un decorador a un `classmethod`.

In [ ]:
def deco(func):
    from functools import wraps
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("decorator called")
        return func(*args, **kwargs)
    return wrapper

class Ejemplo:
    @classmethod
    @deco
    def crear(cls, value):
        print("crear:", cls.__name__, value)
        return cls()

# Llamada:
Ejemplo.crear(10)

Nota: los decoradores se aplican de abajo hacia arriba. En el ejemplo anterior, primero se aplica `deco` al `func`, y luego `classmethod` transforma el resultado en un método de clase.

## 7) Decoradores en propiedades y métodos de clase: ejemplos prácticos

- Validación automática en setters
- Memoización con `cached_property`
- Logging y control de permisos aplicados a métodos de instancia o de clase

In [ ]:
# Ejemplo: control simple de permisos mediante decorador aplicado a método
from functools import wraps

def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(self, *args, **kwargs):
            if getattr(self, "role", None) != role:
                raise PermissionError("Acceso denegado")
            return func(self, *args, **kwargs)
        return wrapper
    return decorator

class Usuario:
    def __init__(self, name, role):
        self.name = name
        self.role = role

    @require_role("admin")
    def eliminar(self, objeto):
        return f"{self.name} eliminó {objeto}"

u1 = Usuario("ana", "user")
u2 = Usuario("luis", "admin")

try:
    print(u1.eliminar('archivo'))
except PermissionError as e:
    print("u1:", e)

print("u2:", u2.eliminar('archivo'))

## 8) Buenas prácticas

- Usar `functools.wraps` en los decoradores para preservar `__name__` y `__doc__`.
- Asegurarse de que el decorador soporte `*args, **kwargs` cuando se aplique a métodos.
- Tener en cuenta el orden de aplicación cuando se combinan `@decorator`, `@classmethod` y `@staticmethod`.